In [ ]:
import numpy as np
import os
import pandas as pd
import pandas_gbq
import matplotlib.pyplot as plt

In [ ]:
# Read in the sample lists

v9_ids = pd.read_csv("/home/jupyter/v9_only.subjects", header=None,dtype=str)[0].tolist()
CDRv9_query_code = "C2025Q4R6"

In [ ]:
def get_peoples_cancer_family_history(data_version: str, cancer_types=["bladder","blood or soft tissue","brain","breast","bone","cervical","head and neck","stomach","other cancer","skin","pancreatic","colon","endocrine","esophagael","eye","kidney","lung","ovarian","thyroid","endometrial","prostate"]):
    query = f"""
    SELECT person_id, answer
    FROM `wb-silky-artichoke-2408.{data_version}.cb_review_survey`
    WHERE LOWER(answer) LIKE "%cancer%" AND LOWER(answer) LIKE "%including%"
    """

    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df = df[~df["answer"].str.contains("Self|non-cancer", na=False)]
    df["family_member"] = df["answer"].str.split("-", n=1).str[1].str.strip()

    # First Degree Relatives
    df_fdr = df[~df["family_member"].isin(["Grandparent", "Grandchild"])]
          
    df_num_fam_count = df[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="num_relatives_w_cancer")
    df_num_fdr_count = df_fdr[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="num_fdrs_w_cancer")

    dfs = []
    for cancer in cancer_types:
        df_cancer = df[df["answer"].str.contains(cancer, case=False, na=False)]
        df_cancer_fdr = df_cancer[~df_cancer["family_member"].isin(["Grandparent","Grandchild"])]

        # All denotated male relatives are also FDRs (e.g., GrandFather or Grandson aren't valid options in the survey)
        if cancer == "breast":
            male_cancer = df_cancer[df_cancer["family_member"].isin(["Father", "Son"])]
            male_cancer_fdr = male_cancer[~male_cancer["family_member"].isin(["Grandparent","Grandchild"])]
    
            df_cancer = df_cancer[~df_cancer["family_member"].isin(["Father", "Son"])]
            df_cancer_fdr = df_cancer_fdr[~df_cancer_fdr["family_member"].isin(["Father", "Son"])]

            # Make specific columns for male breast cancer
            male_relatives = male_cancer[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="male_relatives_w_breast_cancer")
            male_fdrs = male_cancer_fdr[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="male_fdrs_w_breast_cancer")
    
            dfs.append(male_relatives.merge(male_fdrs, on="person_id", how="outer"))
        
        relatives = df_cancer[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name=f"relatives_w_{cancer}_cancer".replace(" ","_"))
        fdrs = df_cancer_fdr[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name=f"fdrs_w_{cancer}_cancer".replace(" ","_"))

        dfs.append(relatives.merge(fdrs, on="person_id", how="outer"))

    result = df_num_fam_count.merge(df_num_fdr_count, on="person_id", how="outer")
    for df_cancer in dfs:
        result = result.merge(df_cancer, on="person_id", how="outer")

    count_cols = [col for col in result.columns if col != "person_id"]
    result[count_cols] = result[count_cols].fillna(0).astype(int)

    return result


df_num_fam_count_v9 = get_peoples_cancer_family_history(data_version = CDRv9_query_code)

In [ ]:
query = """
SELECT
    person_id,
    MAX(start_datetime) AS last_survey_datetime
FROM `wb-silky-artichoke-2408.C2025Q4R6.cb_review_survey`
WHERE LOWER(answer) LIKE "%cancer%"
    AND LOWER(answer) LIKE "%including%"
GROUP BY person_id
ORDER BY last_survey_datetime DESC
"""

df_last_survey = pd.read_gbq(query)
df_last_survey["person_id"] = df_last_survey["person_id"].astype(str)
df_last_survey["last_survey_datetime"] = pd.to_datetime(df_last_survey["last_survey_datetime"])

In [ ]:
def get_person_demographics(data_version:str):
    query = f"""
    SELECT
        st.person_id,
        st.birth_datetime,
        dt_race.concept_name AS race,
        dt_sex.concept_name AS sex_at_birth
    FROM
        `wb-silky-artichoke-2408.{data_version}`.person AS st
    LEFT JOIN
        `wb-silky-artichoke-2408.{data_version}`.concept AS dt_race
        ON dt_race.concept_id = st.race_concept_id
    LEFT JOIN
        `wb-silky-artichoke-2408.{data_version}`.concept AS dt_sex
        ON dt_sex.concept_id = st.sex_at_birth_concept_id
    """

    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    return df

df_demographics_v9 = get_person_demographics(data_version = CDRv9_query_code)

In [ ]:
def get_smoking_frequency(data_version: str):
    query = f"""
    SELECT person_id, question, answer
    FROM `wb-silky-artichoke-2408.{data_version}.cb_review_survey`
    WHERE LOWER(question) LIKE "%smoking: smoke frequency%"
    ORDER BY question
    """
    
    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df = df.rename(columns={"answer": "smoking_frequency"})
    df = df.drop(columns=["question"])

    smoking_map = {"Not At All": 0, "Some Days": 1, "Every Day": 2}
    df["smoking_frequency_scaled"] = df["smoking_frequency"].apply(lambda x: "NA" if pd.isna(x) or "PMI" in str(x) else next((v for k, v in smoking_map.items() if k.lower() in str(x).lower()), pd.NA))
    return df

df_smoking_frequency_v9 = get_smoking_frequency(data_version=CDRv9_query_code)

In [ ]:
def get_alcohol_frequency(data_version: str):
    query = f"""
    SELECT person_id, question, answer
    FROM `wb-silky-artichoke-2408.{data_version}.cb_review_survey`
    WHERE LOWER(question) LIKE "%alcohol: drink frequency past year%"
    ORDER BY question
    """

    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df = df.rename(columns={"answer": "alcohol_frequency"})
    df = df.drop(columns=["question"])

    alcohol_map = {"Never": 0, "Monthly Or Less": 1, "2 to 4 Per Month": 2, "2 to 3 Per Week": 3, "4 or More Per Week": 4}
    df["alcohol_frequency_scaled"] = df["alcohol_frequency"].apply(lambda x: "NA" if pd.isna(x) or "PMI" in str(x) else next((v for k, v in alcohol_map.items() if k.lower() in str(x).lower()), pd.NA))
    return df
    
df_alcohol_frequency_v9 = get_alcohol_frequency(data_version=CDRv9_query_code)

In [ ]:
def return_query(omop_code: int,data_version: str):
    return f"""
    SELECT
        co.person_id,
        co.condition_concept_id,
        MIN(co.condition_start_date) AS condition_start_date
    FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence` co
    JOIN `wb-silky-artichoke-2408.{data_version}.concept_ancestor` ca
        ON co.condition_concept_id = ca.descendant_concept_id
    WHERE ca.ancestor_concept_id = {omop_code}
        AND ca.min_levels_of_separation >= 0
    GROUP BY co.person_id, co.condition_concept_id
    """

def get_descendant_codes(omop_codes,data_version: str):
    codes = ", ".join(map(str, omop_codes))
    query = f"""
    SELECT DISTINCT descendant_concept_id
    FROM `wb-silky-artichoke-2408.{data_version}.concept_ancestor`
    WHERE ancestor_concept_id IN ({codes})
    """
    
    df = pd.read_gbq(query)
    return df["descendant_concept_id"].tolist()

# Define the 12 cancer types and the corresponding codes we will use 443392: "pancancer"
omop_codes = {
    443392: "pancancer",
    4112853: "breast",
    4178976: "thyroid",
    197508: "bladder",
    4038838: "nhl",
    4181351: "ovarian",
    197230: "endometrial",
    196653: "kidney",
    4162276: "melanoma",
    4163261: "prostate",
    37168850: "colon",
    443388: "lung",
    317510: "leukemia",
    4180793: "pancreas",

}

#cutaneous melanoma, non-medullary thyroid cancer, corpus uteri cancer
# no skin cancer of breast, and no uterine cervix cancers, only cutanoues melanomas when possible.
omop_codes_exclude = {
	"breast": [4244051,4247348],
	"endometrial": [196359,441805,198984,4092515],
	"melanoma": [4116085,36712732, 4170619],
    "kidney":[195480,4181357,601140],
    "colon":[1244634,44504337,35624314],
    "lung":[257513],
    "nhl":[4216139,437233,4082311,4149840],
    "ovarian":[196738,4281030,4281027]
}

breast_exclude = omop_codes_exclude["breast"]
endometrial_exclude = omop_codes_exclude["endometrial"]
melanoma_exclude = omop_codes_exclude["melanoma"]
kidney_exclude = omop_codes_exclude["kidney"]
colon_exclude = omop_codes_exclude["colon"]
lung_exclude = omop_codes_exclude["lung"]
nhl_exclude = omop_codes_exclude["nhl"]
ovarian_exclude = omop_codes_exclude["ovarian"]

In [ ]:
def curate_cancer_diagnoses(omop_code):
    cancer_name = omop_codes[omop_code]
    
    if cancer_name == "breast":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv9_query_code))
        df_part2 = pd.read_gbq(return_query(137809, data_version=CDRv9_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    elif cancer_name == "colon":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv9_query_code))
        df_part2 = pd.read_gbq(return_query(36683531, data_version=CDRv9_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    elif cancer_name == "endometrial":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv9_query_code))
        df_part2 = pd.read_gbq(return_query(35206248, data_version=CDRv9_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    else:
        df = pd.read_gbq(return_query(omop_code, data_version=CDRv9_query_code))
        
    df["person_id"] = df["person_id"].astype(str)
    df["condition_start_date"] = pd.to_datetime(df["condition_start_date"])

    exclusion_map = {
        "breast": breast_exclude,
        "endometrial": endometrial_exclude,
        "melanoma": melanoma_exclude,
        "kidney": kidney_exclude,
        "colon": colon_exclude,
        "lung": lung_exclude,
        "nhl": nhl_exclude,
        "ovarian": ovarian_exclude
    }

    if cancer_name in exclusion_map:
        excluded_person_ids = set(df.loc[df["condition_concept_id"].isin(exclusion_map[cancer_name]),"person_id"])

        # Remove Kidney cancer individuals who have both renal pelvis cancer and bladder cancer
        if cancer_name == "kidney":
            bladder_ids = set(pd.read_gbq(return_query(197508, data_version=CDRv9_query_code))["person_id"].astype(str))
            excluded_person_ids &= bladder_ids
            
        df = df[~df["person_id"].isin(excluded_person_ids)]

    df = df.sort_values("condition_start_date",ascending=True).drop_duplicates("person_id", keep="first")
        
    df[f"{cancer_name}_cancer"] = pd.Series(1, index=df.index, dtype=int)
    df[f"{cancer_name}_diagnosis_date"] = df["condition_start_date"]
    df[f"{cancer_name}_concept_id"] = df["condition_concept_id"]
    df = df.drop(columns=["condition_concept_id","condition_start_date"])
    return df

In [ ]:
dfs = []
for omop_code in omop_codes:
    dfs.append(curate_cancer_diagnoses(omop_code))

results = dfs[0]

for df in dfs[1:]:
    results = results.merge(df,on="person_id",how="outer")

In [ ]:
df_merged_v9 = results.merge(df_num_fam_count_v9, on="person_id",how="outer")
df_merged_v9 = pd.merge(df_merged_v9, df_demographics_v9, on="person_id", how="outer")
df_merged_v9 = pd.merge(df_merged_v9, df_smoking_frequency_v9, on="person_id", how="outer")
df_merged_v9 = pd.merge(df_merged_v9, df_alcohol_frequency_v9, on="person_id", how="outer")
df_merged_v9 = df_merged_v9[df_merged_v9['person_id'].isin(v9_ids)]
    
# Impute those with no reported FDR in v7/9with cancer as having 0 FDR cancers
df_merged_v9["num_fdrs_w_cancer"] = df_merged_v9["num_fdrs_w_cancer"].fillna(0).astype(int)

# Make a new column for male breast cancer
df_merged_v9["male_breast_cancer"] = ((df_merged_v9["sex_at_birth"] == "Male") & (df_merged_v9["breast_cancer"] == 1)).astype(int)

# ensure that sex_at_birth is 'Female' for those with breast_cancer, endometrial_cancer, or ovarian_cancer == 1, similalry 'Male' with prostate_cancer == 1
df_merged_v9.loc[df_merged_v9["sex_at_birth"] == "Male", ["breast_cancer", "endometrial_cancer", "ovarian_cancer"]] = 0
df_merged_v9.loc[df_merged_v9["sex_at_birth"] == "Female", "prostate_cancer"] = 0


In [ ]:
def correct_pancancer_flags(df_merged_v9, data_version="C2025Q4R6"):
    query = f"""
    SELECT DISTINCT CAST(co.person_id AS STRING) AS person_id
    FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence` co
    JOIN `wb-silky-artichoke-2408.{data_version}.concept` c
        ON co.condition_concept_id = c.concept_id
    WHERE LOWER(c.concept_name) LIKE '%malignant neoplas%'
    """

    cancer_ids = set(pd.read_gbq(query)["person_id"].astype(str))
    df_merged_v9["person_id"] = df_merged_v9["person_id"].astype(str)

    mask = df_merged_v9["person_id"].isin(cancer_ids)
    previously_unflagged = mask & df_merged_v9["pancancer_cancer"].fillna(0).eq(0)

    df_merged_v9.loc[mask, "pancancer_cancer"] = 1

    print(f"Individuals with matching diagnoses in df_merged_v9: {mask.sum():,}")
    print(f"Previously unflagged individuals corrected: {previously_unflagged.sum():,}")

    return df_merged_v9

df_merged_v9 = correct_pancancer_flags(df_merged_v9)

In [ ]:
# def get_excluded_person_ids(omop_codes: dict, data_version: str):
#     codes = [code for code_list in omop_codes.values() for code in code_list]
#     codes_str = ", ".join(map(str, codes))

#     query = f"""
#     SELECT DISTINCT co.person_id
#     FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence` co
#     JOIN `wb-silky-artichoke-2408.{data_version}.concept_ancestor` ca
#         ON co.condition_concept_id = ca.descendant_concept_id
#     WHERE ca.ancestor_concept_id IN ({codes_str})
#     """

#     df = pd.read_gbq(query)
#     return set(df["person_id"].astype(str))

In [ ]:
def curate_cancer_diagnoses(omop_code):
    cancer_name = omop_codes[omop_code]
    
    if cancer_name == "breast":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv9_query_code))
        df_part2 = pd.read_gbq(return_query(137809, data_version=CDRv9_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    elif cancer_name == "colon":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv9_query_code))
        df_part2 = pd.read_gbq(return_query(36683531, data_version=CDRv9_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    else:
        df = pd.read_gbq(return_query(omop_code, data_version=CDRv9_query_code))
        
    df["person_id"] = df["person_id"].astype(str)
    df["condition_start_date"] = pd.to_datetime(df["condition_start_date"])
    
    
    if cancer_name == "breast":
        df = df[~df["condition_concept_id"].isin(breast_exclude)]
    elif cancer_name == "endometrial":
        df = df[~df["condition_concept_id"].isin(endometrial_exclude)]
    elif cancer_name == "melanoma":
        df = df[~df["condition_concept_id"].isin(melanoma_exclude)]
    elif cancer_name == "kidney":
        df = df[~df["condition_concept_id"].isin(kidney_exclude)]
    elif cancer_name == "colon":
        df = df[~df["condition_concept_id"].isin(colon_exclude)]
    elif cancer_name == "lung":
        df = df[~df["condition_concept_id"].isin(lung_exclude)]
    elif cancer_name == "nhl":
        df = df[~df["condition_concept_id"].isin(nhl_exclude)]

    df = df.sort_values("condition_start_date",ascending=True).drop_duplicates("person_id", keep="first")
        
    df[f"{cancer_name}_cancer"] = pd.Series(1, index=df.index, dtype=int)
    df[f"{cancer_name}_diagnosis_date"] = df["condition_start_date"]
    df[f"{cancer_name}_concept_id"] = df["condition_concept_id"]
    df = df.drop(columns=["condition_concept_id","condition_start_date"])
    return df

In [ ]:
dfs = []
for omop_code in omop_codes:
    dfs.append(curate_cancer_diagnoses(omop_code))

results = dfs[0]

for df in dfs[1:]:
    results = results.merge(df,on="person_id",how="outer")

In [ ]:
df_merged_v9 = results.merge(df_num_fam_count_v9, on="person_id",how="outer")
df_merged_v9 = pd.merge(df_merged_v9, df_demographics_v9, on="person_id", how="outer")
df_merged_v9 = pd.merge(df_merged_v9, df_smoking_frequency_v9, on="person_id", how="outer")
df_merged_v9 = pd.merge(df_merged_v9, df_alcohol_frequency_v9, on="person_id", how="outer")
df_merged_v9 = df_merged_v9[df_merged_v9['person_id'].isin(v9_ids)]

# Remove Related Samples
related_ids = set(pd.read_csv("relatedness_flagged_samples.tsv", sep="\t", header=None, dtype=str)[0].str.strip())
df_merged_v9 = df_merged_v9[~df_merged_v9["person_id"].astype(str).isin(related_ids)].copy()

# Read in pcs
pcs_df = pd.read_csv("gs://dataproc-staging-wb-cordial-diamond-9893/UFC_REFERENCE_FILES/AoU_CDRv9_pcs.tsv.gz",sep='\t',index_col=False,dtype={"person_id":str})
pcs_df = pcs_df.drop(columns=[f"PC{i}" for i in range(6, 11)])
df_merged_v9 = df_merged_v9.merge(pcs_df,how="left",on="person_id")

# Impute those with no reported FDR in v7/9with cancer as having 0 FDR cancers
df_merged_v9["num_fdrs_w_cancer"] = df_merged_v9["num_fdrs_w_cancer"].fillna(0).astype(int)

# Make a new column for male breast cancer
df_merged_v9["male_breast_cancer"] = ((df_merged_v9["sex_at_birth"] == "Male") & (df_merged_v9["breast_cancer"] == 1)).astype(int)

# ensure that sex_at_birth is 'Female' for those with breast_cancer, endometrial_cancer, or ovarian_cancer == 1, similalry 'Male' with prostate_cancer == 1
df_merged_v9.loc[df_merged_v9["sex_at_birth"] == "Male", ["breast_cancer", "endometrial_cancer", "ovarian_cancer"]] = 0
df_merged_v9.loc[df_merged_v9["sex_at_birth"] == "Female", "prostate_cancer"] = 0


In [ ]:
def correct_pancancer_flags(df_merged_v9, data_version="C2025Q4R6"):
    query = f"""
    SELECT DISTINCT CAST(co.person_id AS STRING) AS person_id
    FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence` co
    JOIN `wb-silky-artichoke-2408.{data_version}.concept` c
        ON co.condition_concept_id = c.concept_id
    WHERE LOWER(c.concept_name) LIKE '%malignant neoplas%'
    """

    cancer_ids = set(pd.read_gbq(query)["person_id"].astype(str))
    df_merged_v9["person_id"] = df_merged_v9["person_id"].astype(str)

    mask = df_merged_v9["person_id"].isin(cancer_ids)
    previously_unflagged = mask & df_merged_v9["pancancer_cancer"].fillna(0).eq(0)

    df_merged_v9.loc[mask, "pancancer_cancer"] = 1

    print(f"Individuals with matching diagnoses in df_merged_v9: {mask.sum():,}")
    print(f"Previously unflagged individuals corrected: {previously_unflagged.sum():,}")

    return df_merged_v9

df_merged_v9 = correct_pancancer_flags(df_merged_v9)

In [ ]:
def add_last_recorded_ehr(df):
    data_version = "C2025Q4R6"
    query = f"""
    SELECT
        CAST(person_id AS STRING) AS person_id,
        MAX(condition_start_date) AS last_recorded_ehr
    FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence`
    GROUP BY person_id
    """
    last_dates = pd.read_gbq(query)
    df = df.copy()
    df["person_id"] = df["person_id"].astype(str)
    df = df.merge(last_dates, on="person_id", how="left")
    df["last_recorded_ehr"] = pd.to_datetime(df["last_recorded_ehr"])
    return df

df_merged_v9 = add_last_recorded_ehr(df_merged_v9)

In [ ]:
df_merged_v9 = df_merged_v9[~(df_merged_v9["pancancer_cancer"].fillna(0).eq(0) & df_merged_v9["last_recorded_ehr"].isna())].copy()

In [ ]:
def add_age(df):
    df = df.copy()
    for col in ["birth_datetime", "pancancer_diagnosis_date", "last_recorded_ehr"]:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True).dt.tz_localize(None)

    is_case = df["pancancer_cancer"].eq(1)
    reference_date = df["last_recorded_ehr"].where(~is_case, df["pancancer_diagnosis_date"])
    df["age"] = ((reference_date - df["birth_datetime"]).dt.days / 365.2425).where(reference_date.notna() & df["birth_datetime"].notna())

    print(f"Cases missing age: {df.loc[is_case, 'age'].isna().sum():,} / {is_case.sum():,}")
    print(f"Controls missing age: {df.loc[~is_case, 'age'].isna().sum():,} / {(~is_case).sum():,}")

    return df

df_merged_v9 = add_age(df_merged_v9)

In [ ]:
pgv_carriers = pd.read_csv(
    "gs://dataproc-staging-wb-cordial-diamond-9893/UFC_REFERENCE_FILES/oct11_2026_pgv_carriers.CDRv9.list.tsv",
    sep="\t",
    header=None,
    dtype=str
)[0].str.strip()

# First exclude all PGV carriers
replication_cohort = df_merged_v9[
    ~df_merged_v9["person_id"].astype(str).str.strip().isin(pgv_carriers)
].copy()

# Then filter for cases and controls
replication_cohort = replication_cohort[
    (
        (replication_cohort["pancancer_cancer"] == 1)
        & (replication_cohort["num_relatives_w_cancer"] >= 3)
    )
    | (
        (replication_cohort["pancancer_cancer"].fillna(0) == 0)
        & (replication_cohort["num_relatives_w_cancer"].fillna(0) == 0)
    )
].copy()

In [ ]:
# # # Save files locally, then copy to gsutil
# df_merged_v9.to_csv("CDRv9.metadata",sep='\t',index=False)
# !gzip CDRv9.metadata
# !gsutil -m cp CDRv9.metadata.gz gs://dataproc-staging-wb-cordial-diamond-9893/UFC_REFERENCE_FILES/

# replication_cohort.to_csv("CDRv9.ufc_replication.metadata",sep='\t',index=False)
# !gzip CDRv9.ufc_replication.metadata
# !gsutil -m cp CDRv9.ufc_replication.metadata.gz gs://dataproc-staging-wb-cordial-diamond-9893/UFC_REFERENCE_FILES/